# 009 — Qualification des plantes par lots
Aucun appel API. Les profils et revues sont fournis manuellement. Consensus provisoire, non expert.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "gdai-plant").is_dir())
PLANT = ROOT / "gdai-plant"
sys.path.insert(0,str(PLANT))
from enrichment.prepare_profile_pilot import prepare, with_sources
from enrichment.prepare_claude_review import prepare as review_prompt
from enrichment.qualify_batches import qualify
CATALOGUE = PLANT / "data/plants-mvp-100.json"
WORK = PLANT / "data/enrichment/mvp-100"
read = lambda p: json.loads(p.read_text(encoding="utf-8"))
catalogue = read(CATALOGUE)
assert len(catalogue) == len({p["scientific_name"] for p in catalogue}) == 100
protocol = (PLANT/"prompts/horticultural-common-v1.md").read_text()
protocol += "\nCatalogue listing suffices; do not require current stock or price.\n"
protocol += (PLANT/"schemas/plant-enrichment-v1.schema.json").read_text()
protocol = with_sources(protocol, read(PLANT/"config/gardnaize-horticultural-sources-v1.json"))
print(len(catalogue), "candidate plants")

In [ ]:
# Prepare prompts and reuse existing pilot without overwriting files.
import shutil
for offset in range(0,len(catalogue),5):
    folder = WORK / f"batch-{offset//5+1:02d}"
    folder.mkdir(parents=True,exist_ok=True)
    prompt = folder/"gemini-prompt.txt"
    if not prompt.exists():
        prompt.write_text(prepare(catalogue,protocol,offset,min(5,len(catalogue)-offset)),encoding="utf-8")
for src,dst in [
    (PLANT/"data/enrichment/pilot-05-recovered/profiles-unvalidated.json",WORK/"batch-01/profiles.json"),
    (PLANT/"data/enrichment/claude/pilot-05-review-v1.json",WORK/"batch-01/review.json")]:
    if src.exists() and not dst.exists():
        shutil.copyfile(src,dst)
print("Prompts ready:", WORK)

In [ ]:
# Save Gemini JSON as profiles.json in each batch folder.
# Run this cell to generate the corresponding Claude prompt.
batches = []
progress = []
for offset in range(0,len(catalogue),5):
    folder = WORK / f"batch-{offset//5+1:02d}"
    names = [p["scientific_name"] for p in catalogue[offset:offset+5]]
    source, review = folder/"profiles.json", folder/"review.json"
    if source.exists():
        payload = read(source)
        prompt = review_prompt(payload,names)
        dest = folder/"claude-prompt.txt"
        if not dest.exists():
            dest.write_text(prompt,encoding="utf-8")
        elif dest.read_text(encoding="utf-8") != prompt:
            raise ValueError(f"Stale review prompt: {dest}")
        if review.exists():
            batches.append((names,payload,read(review)))
    progress.append({"batch":folder.name,"profiles":source.exists(),"review":review.exists()})
display(pd.DataFrame(progress))
profiles, quality, report = qualify(catalogue,batches)
display(pd.DataFrame(quality["plants"]))
print(report["processed"],"/",report["expected"],report["quality_counts"])

In [ ]:
# Export only when all candidate plants have been reviewed.
assert report["processed"] == len(catalogue) and not report["missing"]
OUT = WORK/"qualified-v1"
OUT.mkdir(exist_ok=False)
for name,data in [("profiles-provisional.json",profiles),("quality.json",quality),("qualification-report.json",report)]:
    (OUT/name).write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding="utf-8")
print(OUT)